# VisionInspect — 03 Train YOLO Segmentation

Baseline YOLO instance-segmentation training for the three prepared industrial inspection datasets:

- DAGM
- KolektorSDD2
- Magnetic Tile

Each dataset remains a **separate experiment** because their class semantics and visual domains differ.

### Objectives
1. Validate the generated dataset YAML files.
2. Train one reproducible YOLO segmentation baseline per dataset.
3. Track configuration, metrics and artifacts with MLflow.
4. Evaluate the selected model on the held-out test set.
5. Produce a compact baseline comparison for the portfolio.

This notebook intentionally avoids hyperparameter sweeps. Establish the baseline first; optimize later.


In [1]:
# Install once if needed:
# %pip install -U ultralytics mlflow pyyaml pandas matplotlib

from pathlib import Path
import platform
import subprocess
import time

import pandas as pd
import yaml
import mlflow
from ultralytics import YOLO

SEED = 42

DATA_ROOT = Path("datasets")
YOLO_ROOT = DATA_ROOT / "yolo_segmentation"
MLFLOW_DIR = Path("mlruns")
RUNS_DIR = Path("runs") / "segment"

PROJECT_NAME = "VisionInspect"

DATASETS = {
    "DAGM": YOLO_ROOT / "DAGM" / "dataset.yaml",
    "KolektorSDD2": YOLO_ROOT / "KolektorSDD2" / "dataset.yaml",
    "Magnetic_Tile": YOLO_ROOT / "Magnetic_Tile" / "dataset.yaml",
}

# Baseline model. Change deliberately, not during comparison.
MODEL_NAME = "yolo26n-seg.pt"

EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 16
WORKERS = 4
DEVICE = None  # None=Ultralytics auto; 0=first CUDA GPU; "cpu"=CPU

print("Dataset root:", DATA_ROOT.resolve())
print("YOLO root:", YOLO_ROOT.resolve())
print("Model:", MODEL_NAME)


/home/zubair/AI Development Directory/anomaly-detection/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/zubair/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Dataset root: /home/zubair/AI Development Directory/anomaly-detection/datasets
YOLO root: /home/zubair/AI Development Directory/anomaly-detection/datasets/yolo_segmentation
Model: yolo26n-seg.pt


## 1. Dataset YAML validation

The notebook expects the three generated YAML files at:

```text
datasets/yolo_segmentation/
├── DAGM/dataset.yaml
├── KolektorSDD2/dataset.yaml
└── Magnetic_Tile/dataset.yaml
```

Empty label files are valid: they represent normal images with no annotated defect.


In [2]:
def load_dataset_yaml(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing dataset YAML: {path}")
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def resolve_dataset_root(yaml_path, cfg):
    root = Path(cfg["path"])
    return root if root.is_absolute() else (Path(yaml_path).parent / root).resolve()

dataset_info = {}

for name, yaml_path in DATASETS.items():
    cfg = load_dataset_yaml(yaml_path)
    root = resolve_dataset_root(yaml_path, cfg)
    dataset_info[name] = {"yaml": yaml_path, "config": cfg, "root": root}

    print(f"\n{name}")
    print("root   :", root)
    print("names  :", cfg.get("names"))
    print("train  :", cfg.get("train"))
    print("val    :", cfg.get("val"))
    print("test   :", cfg.get("test"))



DAGM
root   : /home/zubair/AI Development Directory/anomaly-detection/datasets/yolo_segmentation/DAGM
names  : {0: 'Class1', 1: 'Class2', 2: 'Class3', 3: 'Class4', 4: 'Class5', 5: 'Class6', 6: 'Class7', 7: 'Class8', 8: 'Class9', 9: 'Class10'}
train  : images/train
val    : images/val
test   : images/test

KolektorSDD2
root   : /home/zubair/AI Development Directory/anomaly-detection/datasets/yolo_segmentation/KolektorSDD2
names  : {0: 'defect'}
train  : images/train
val    : images/val
test   : images/test

Magnetic_Tile
root   : /home/zubair/AI Development Directory/anomaly-detection/datasets/yolo_segmentation/Magnetic_Tile
names  : {0: 'Blowhole', 1: 'Break', 2: 'Crack', 3: 'Fray', 4: 'Uneven'}
train  : images/train
val    : images/val
test   : images/test


## 2. Dataset integrity checks

In [3]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def inspect_split(root, split):
    image_dir = root / "images" / split
    label_dir = root / "labels" / split

    if not image_dir.exists():
        return {"split": split, "images": 0, "labels": 0,
                "missing_labels": 0, "empty_labels": 0}

    images = [p for p in image_dir.iterdir()
              if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
    labels = {p.stem: p for p in label_dir.glob("*.txt")} if label_dir.exists() else {}

    missing = 0
    empty = 0
    for image in images:
        label = labels.get(image.stem)
        if label is None:
            missing += 1
        elif not label.read_text(encoding="utf-8").strip():
            empty += 1

    return {"split": split, "images": len(images), "labels": len(labels),
            "missing_labels": missing, "empty_labels": empty}

rows = []
for name, info in dataset_info.items():
    for split in ["train", "val", "test"]:
        r = inspect_split(info["root"], split)
        r["dataset"] = name
        rows.append(r)

integrity_df = pd.DataFrame(rows)[
    ["dataset", "split", "images", "labels", "missing_labels", "empty_labels"]
]
display(integrity_df)


,dataset,split,images,labels,missing_labels,empty_labels
0,DAGM,train,1725,1725,0,1478
1,DAGM,val,911,911,0,793
2,DAGM,test,1150,1150,0,1000
3,KolektorSDD2,train,2332,2332,0,2086
4,KolektorSDD2,val,0,0,0,0
5,KolektorSDD2,test,1004,1004,0,894
6,Magnetic_Tile,train,940,940,0,669
7,Magnetic_Tile,val,0,0,0,0
8,Magnetic_Tile,test,404,404,0,287


## 3. MLflow setup

MLflow is used for experiment tracking. It is deliberately separate from serving.

Each dataset receives its own experiment:

```text
VisionInspect_DAGM
VisionInspect_KolektorSDD2
VisionInspect_Magnetic_Tile
```


In [8]:
import os

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
mlflow.set_tracking_uri("sqlite:///mlflow.db")

for dataset_name in DATASETS:
    mlflow.set_experiment(f"{PROJECT_NAME}_{dataset_name}")

print("MLflow tracking URI:", mlflow.get_tracking_uri())

2026/08/11 21:27:07 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/11 21:27:07 INFO mlflow.store.db.utils: Updating database tables
2026/08/11 21:27:10 INFO mlflow.tracking.fluent: Experiment with name 'VisionInspect_DAGM' does not exist. Creating a new experiment.
2026/08/11 21:27:10 INFO mlflow.tracking.fluent: Experiment with name 'VisionInspect_KolektorSDD2' does not exist. Creating a new experiment.
2026/08/11 21:27:10 INFO mlflow.tracking.fluent: Experiment with name 'VisionInspect_Magnetic_Tile' does not exist. Creating a new experiment.


MLflow tracking URI: sqlite:///mlflow.db


## 4. Reproducibility / environment metadata

In [13]:
def git_commit():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            stderr=subprocess.DEVNULL, text=True
        ).strip()
    except Exception:
        return "not_available"

ENVIRONMENT = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "git_commit": git_commit(),
}

try:
    import torch
    ENVIRONMENT["torch"] = torch.__version__
    ENVIRONMENT["cuda_available"] = str(torch.cuda.is_available())
    ENVIRONMENT["cuda_devices"] = str(torch.cuda.device_count())
    if torch.cuda.is_available():
        ENVIRONMENT["gpu"] = torch.cuda.get_device_name(0)
except Exception as exc:
    ENVIRONMENT["torch"] = f"unavailable: {exc}"

ENVIRONMENT


{'python': '3.14.4',
 'platform': 'Linux-7.0.0-29-generic-x86_64-with-glibc2.43',
 'git_commit': 'not_available',
 'torch': '2.13.0+cu130',
 'cuda_available': 'True',
 'cuda_devices': '1',
 'gpu': 'NVIDIA GeForce RTX 5070 Laptop GPU'}

## 5. Baseline configuration

Keep this fixed across the three datasets so the comparison is meaningful.

The important rule is: **baseline first, tuning second**.


In [14]:
TRAIN_CONFIG = {
    "epochs": EPOCHS,
    "imgsz": IMAGE_SIZE,
    "batch": BATCH_SIZE,
    "workers": WORKERS,
    "seed": SEED,
    "patience": 20,
    "optimizer": "auto",
    "pretrained": True,
    "plots": True,
    "verbose": True,
}

TRAIN_CONFIG


{'epochs': 100,
 'imgsz': 640,
 'batch': 16,
 'workers': 4,
 'seed': 42,
 'patience': 20,
 'optimizer': 'auto',
 'pretrained': True,
 'plots': True,
 'verbose': True}

## 6. Metric extraction

In [10]:
def as_float(value):
    try:
        return float(value)
    except Exception:
        return None

def extract_metrics(metrics):
    out = {}

    if hasattr(metrics, "box"):
        for key, attr in [
            ("precision", "mp"),
            ("recall", "mr"),
            ("map50", "map50"),
            ("map50_95", "map"),
        ]:
            value = getattr(metrics.box, attr, None)
            if value is not None:
                out[f"box_{key}"] = as_float(value)

    if hasattr(metrics, "seg"):
        for key, attr in [
            ("precision", "mp"),
            ("recall", "mr"),
            ("map50", "map50"),
            ("map50_95", "map"),
        ]:
            value = getattr(metrics.seg, attr, None)
            if value is not None:
                out[f"seg_{key}"] = as_float(value)

    return out


## 7. Train one dataset

The function trains the model, logs parameters and environment metadata to MLflow, evaluates validation performance, and logs the native Ultralytics artifacts.

The test set is **not** used here.


In [11]:
def train_dataset(dataset_name):
    yaml_path = DATASETS[dataset_name]
    experiment_name = f"{PROJECT_NAME}_{dataset_name}"

    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment is None:
        mlflow.set_experiment(experiment_name)
        experiment = mlflow.get_experiment_by_name(experiment_name)

    with mlflow.start_run(
        experiment_id=experiment.experiment_id,
        run_name=f"{dataset_name}_baseline",
    ) as run:

        cfg = dataset_info[dataset_name]["config"]

        mlflow.log_params({
            "model": MODEL_NAME,
            "dataset": dataset_name,
            "epochs": EPOCHS,
            "imgsz": IMAGE_SIZE,
            "batch": BATCH_SIZE,
            "workers": WORKERS,
            "seed": SEED,
            "patience": TRAIN_CONFIG["patience"],
        })
        mlflow.log_dict(cfg, "dataset.yaml")
        mlflow.log_dict(ENVIRONMENT, "environment.json")

        model = YOLO(MODEL_NAME)
        start = time.time()

        model.train(
            data=str(yaml_path),
            project=str(RUNS_DIR),
            name=dataset_name.lower(),
            exist_ok=True,
            device=DEVICE,
            **TRAIN_CONFIG,
        )

        train_seconds = time.time() - start
        mlflow.log_metric("training_seconds", train_seconds)

        best_model = Path(model.trainer.best)
        last_model = Path(model.trainer.last)
        native_dir = Path(model.trainer.save_dir)

        for path in [best_model, last_model]:
            if path.exists():
                mlflow.log_artifact(str(path), artifact_path="model")

        if native_dir.exists():
            mlflow.log_artifacts(str(native_dir), artifact_path="ultralytics_run")

        val = model.val(
            data=str(yaml_path),
            split="val",
            imgsz=IMAGE_SIZE,
            batch=BATCH_SIZE,
            device=DEVICE,
            plots=True,
        )

        val_metrics = extract_metrics(val)
        if val_metrics:
            mlflow.log_metrics({
                f"val_{k}": v for k, v in val_metrics.items()
                if v is not None
            })

        mlflow.set_tags({
            "project": PROJECT_NAME,
            "dataset": dataset_name,
            "task": "instance_segmentation",
            "stage": "baseline",
        })

        return {
            "run_id": run.info.run_id,
            "best_model": str(best_model),
            "last_model": str(last_model),
            "native_run_dir": str(native_dir),
            "training_seconds": train_seconds,
            "val_metrics": val_metrics,
        }


## 8. Run the first baseline

Start with **Magnetic Tile** because it is small and gives the fastest end-to-end validation.

If this completes successfully, run KolektorSDD2 and DAGM.


In [16]:
# Uncomment to run:

baseline_results = {}
baseline_results["Magnetic_Tile"] = train_dataset("Magnetic_Tile")
baseline_results["Magnetic_Tile"]["val_metrics"]


Ultralytics 8.4.117 🚀 Python-3.14.4 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 7708MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/yolo_segmentation/Magnetic_Tile/dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.0, mu

2026/08/11 21:33:51 INFO mlflow.tracking.fluent: Experiment with name 'runs/segment' does not exist. Creating a new experiment.


MLflow: logging run_id(e271a4c46c944511a58c8d92875f6944) to sqlite:///mlflow.db
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to /home/zubair/AI Development Directory/anomaly-detection/runs/segment/runs/segment/magnetic_tile
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
      1/100      3.16G      1.496      3.361      22.95    0.01131      8.381          8        640: 100% ━━━━━━━━━━━━ 59/59 4.5it/s 13.1s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.7it/s 4.2s0.7ss
                   all        202         78   0.000678      0.314     0.0606     0.0298   0.000758      0.389     0.0639     0.0307

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
      2/1

{'box_precision': 0.8363833284662896,
 'box_recall': 0.7720490164948297,
 'box_map50': 0.8464394251042195,
 'box_map50_95': 0.5335586360601706,
 'seg_precision': 0.8621853535608359,
 'seg_recall': 0.6910998124446573,
 'seg_map50': 0.8188881898048079,
 'seg_map50_95': 0.46018367816984557}

## 9. Run the remaining baselines

In [19]:
# After Magnetic Tile succeeds:

baseline_results["KolektorSDD2"] = train_dataset("KolektorSDD2")

Ultralytics 8.4.117 🚀 Python-3.14.4 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 7708MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/yolo_segmentation/KolektorSDD2/dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.0, mul

In [20]:
baseline_results["DAGM"] = train_dataset("DAGM")

Ultralytics 8.4.117 🚀 Python-3.14.4 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 7708MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/yolo_segmentation/DAGM/dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale

## 10. Held-out test evaluation

Run this only after selecting the best checkpoint from validation.

The test set is evaluated once for the final baseline report.


In [ ]:
def evaluate_test(dataset_name, best_model_path):
    yaml_path = DATASETS[dataset_name]

    experiment = mlflow.get_experiment_by_name(
        f"{PROJECT_NAME}_{dataset_name}"
    )

    with mlflow.start_run(
        experiment_id=experiment.experiment_id,
        run_name=f"{dataset_name}_test_evaluation",
    ):
        mlflow.log_params({
            "dataset": dataset_name,
            "split": "test",
            "model": str(best_model_path),
        })

        model = YOLO(best_model_path)

        metrics = model.val(
            data=str(yaml_path),
            split="test",
            imgsz=IMAGE_SIZE,
            batch=BATCH_SIZE,
            device=DEVICE,
            plots=True,
        )

        test_metrics = extract_metrics(metrics)

        if test_metrics:
            mlflow.log_metrics({
                f"test_{k}": v for k, v in test_metrics.items()
                if v is not None
            })

        mlflow.set_tags({
            "project": PROJECT_NAME,
            "dataset": dataset_name,
            "task": "instance_segmentation",
            "stage": "test_evaluation",
        })

        return test_metrics


In [22]:

# Example:
test_metrics = evaluate_test(
    "Magnetic_Tile",
    baseline_results["Magnetic_Tile"]["best_model"],
)
test_metrics


Ultralytics 8.4.117 🚀 Python-3.14.4 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 7708MiB)
YOLO26n-seg summary (fused): 139 layers, 2,689,859 parameters, 0 gradients, 9.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 192.3±61.2 MB/s, size: 21.7 KB)
val: Scanning /home/zubair/AI Development Directory/anomaly-detection/datasets/yolo_segmentation/Magnetic_Tile/labels/test... 202 images, 144 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 202/202 5.1Kit/s 0.0s
val: New cache created: /home/zubair/AI Development Directory/anomaly-detection/datasets/yolo_segmentation/Magnetic_Tile/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 7.0it/s 1.9s<0.1s
                   all        202         69      0.896      0.717      0.849       0.57       0.74      0.811       0.83      0.461
              Blowhole         17         17      0.969          1   

{'box_precision': 0.8961672002012113,
 'box_recall': 0.7174882856470528,
 'box_map50': 0.8489513613579405,
 'box_map50_95': 0.569656524108495,
 'seg_precision': 0.7400502482090943,
 'seg_recall': 0.8112925131881956,
 'seg_map50': 0.8296373168884779,
 'seg_map50_95': 0.4608649352595569}

## 11. Qualitative prediction inspection

Industrial inspection should not be judged by mAP alone.

Inspect:

- normal images → no false detections
- tiny defects
- low-contrast defects
- irregular defects
- multiple disconnected defects
- difficult DAGM classes, especially Classes 7–10


In [34]:
def predict_samples(dataset_name, model_path, split="test", n=6, conf=0.25):
    root = dataset_info[dataset_name]["root"]
    image_dir = root / "images" / split

    images = sorted([
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ])[:n]

    model = YOLO(model_path)

    results = model.predict(
        source=[str(p) for p in images],
        conf=conf,
        imgsz=IMAGE_SIZE,
        device=DEVICE,
        save=True,
        project=str(RUNS_DIR / "predictions"),
        name=dataset_name.lower(),
        exist_ok=True,
    )

    print("Predictions:", RUNS_DIR / "predictions" / dataset_name.lower())
    return results

# Example:
# predict_samples(
#     "Magnetic_Tile",
#     baseline_results["Magnetic_Tile"]["best_model"],
#     split="test",
# )

# predict_samples(
#     "DAGM",
#     baseline_results["DAGM"]["best_model"],
#     split="test",
# )

predict_samples(
    "KolektorSDD2",
    baseline_results["KolektorSDD2"]["best_model"],
    split="test",
)




0: 640x640 (no detections), 1.9ms
1: 640x640 (no detections), 1.9ms
2: 640x640 (no detections), 1.9ms
3: 640x640 (no detections), 1.9ms
4: 640x640 (no detections), 1.9ms
5: 640x640 (no detections), 1.9ms
Speed: 0.7ms preprocess, 1.9ms inference, 0.1ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /home/zubair/AI Development Directory/anomaly-detection/runs/segment/runs/segment/predictions/kolektorsdd2
Predictions: runs/segment/predictions/kolektorsdd2


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 depth: None
 keypoints: None
 masks: None
 names: {0: 'defect'}
 obb: None
 orig_img: array([[[42, 29, 24],
         [38, 30, 32],
         [47, 38, 40],
         ...,
         [47, 41, 43],
         [41, 40, 38],
         [45, 46, 54]],
 
        [[27, 24, 23],
         [26, 21, 23],
         [32, 29, 30],
         ...,
         [41, 40, 41],
         [47, 46, 45],
         [42, 49, 53]],
 
        [[22, 16, 16],
         [20, 18, 19],
         [27, 24, 26],
         ...,
         [38, 41, 45],
         [46, 52, 54],
         [51, 51, 62]],
 
        ...,
 
        [[ 3,  6,  7],
         [17, 17, 22],
         [40, 38, 42],
         ...,
         [39, 39, 37],
         [37, 31, 32],
         [36, 32, 33]],
 
        [[ 6,  6,  5],
         [13, 11, 13],
         [27, 26, 28],
         ...,
         [42, 40, 40],
         [32, 32, 33],
         [37, 36, 32]],
 
        [[ 3, 

## 12. Baseline comparison

Use the segmentation metrics as the primary model-selection metrics:

- **Segmentation mAP50**
- **Segmentation mAP50-95**
- Precision
- Recall
- Training time

Do not interpret the three datasets as one homogeneous benchmark. They have different domains, defect types and difficulty.


In [ ]:
# Fill from the completed MLflow runs after test evaluation.

comparison = pd.DataFrame([
    {
        "dataset": "DAGM",
        "seg_mAP50": ...,
        "seg_mAP50_95": ...,
        "precision": ...,
        "recall": ...,
        "training_seconds": ...,
    },
    {
        "dataset": "KolektorSDD2",
        "seg_mAP50": ...,
        "seg_mAP50_95": ...,
        "precision": ...,
        "recall": ...,
        "training_seconds": ...,
    },
    {
        "dataset": "Magnetic_Tile",
        "seg_mAP50": ...,
        "seg_mAP50_95": ...,
        "precision": ...,
        "recall": ...,
        "training_seconds": ...,
    },
])

display(comparison)


,dataset,seg_mAP50,seg_mAP50_95,precision,recall,training_seconds
0,DAGM,Ellipsis,Ellipsis,Ellipsis,Ellipsis,Ellipsis
1,KolektorSDD2,Ellipsis,Ellipsis,Ellipsis,Ellipsis,Ellipsis
2,Magnetic_Tile,Ellipsis,Ellipsis,Ellipsis,Ellipsis,Ellipsis


: 

In [22]:
from ultralytics import YOLO
from pathlib import Path

BASE_RUNS_DIR = Path("/home/zubair/AI Development Directory/anomaly-detection/runs/segment/runs/segment")
# output_dir = Path("/home/zubair/AI Development Directory/anomaly-detection/runs/segment/runs/segment")??/


import shutil
from pathlib import Path
from ultralytics import YOLO

def export_yolo_model_to_onnx(best_model_path, output_dir):
    model = YOLO(best_model_path)
    
    # 1. Export (Ultralytics returns the file path of the generated ONNX model)
    exported_path_str = model.export(format="onnx", dynamic=True, simplify=True)
    exported_file = Path(exported_path_str)
    
    # 2. Move or rename to your desired target destination
    target_path = Path(output_dir) / exported_file.name
    target_path.parent.mkdir(parents=True, exist_ok=True)
    
    shutil.move(exported_file, target_path)
    print(f"Model successfully saved to {target_path}")

In [23]:
# export the best models for each dataset to ONNX format, path to best models in BASE_RUNS_DIR
for dataset_name in ["dagm", "kolektorsdd2", "magnetic_tile"]:
    best_model_path = BASE_RUNS_DIR / dataset_name.lower() / "weights" / "best.pt"
    output_dir = BASE_RUNS_DIR / dataset_name.lower() / "onnx_models"
    export_yolo_model_to_onnx(best_model_path, output_dir)

Ultralytics 8.4.117 🚀 Python-3.14.4 torch-2.13.0+cu130 CPU (AMD Ryzen AI 9 365 w/ Radeon 880M)
YOLO26n-seg summary (fused): 139 layers, 2,690,834 parameters, 0 gradients, 9.1 GFLOPs

PyTorch: starting from '/home/zubair/AI Development Directory/anomaly-detection/runs/segment/runs/segment/dagm/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 300, 38), (1, 32, 160, 160)) (6.3 MB)

ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 2.0s, saved as '/home/zubair/AI Development Directory/anomaly-detection/runs/segment/runs/segment/dagm/weights/best.onnx' (10.9 MB)

Export complete (2.2s)
Results saved to /home/zubair/AI Development Directory/anomaly-detection/runs/segment/runs/segment/dagm/weights/best.onnx
Predict:         yolo predict task=segment model=/home/zubair/AI Development Directory/anomaly-detection/runs/segment/runs/segment/dagm/weights/best.onnx imgsz=640 
Validate:        yolo val task

# Expected output

```text
VisionInspect/
├── datasets/
│   └── yolo_segmentation/
│
├── runs/
│   └── segment/
│       ├── magnetic_tile/
│       ├── kolektorsdd2/
│       ├── dagm/
│       └── predictions/
│
├── mlruns/
│
└── notebooks/
    ├── 01_dataset_analysis.ipynb
    ├── 02_prepare_yolo_segmentation.ipynb
    └── 03_train_yolo_segmentation.ipynb
```

### Next stage

After the baselines are stable:

```text
YOLO Segmentation
       │
       ├── PyTorch checkpoint
       │
       └── ONNX
             │
             ├── FastAPI inference service
             │
             └── Streamlit inspection UI
```

Triton comes later, after ONNX inference is working and benchmarked. It is an optimization/serving component, not a prerequisite for establishing the baseline.
